## Assignment - 2

1.   Name - Atish Kadam
2.   Roll No - CS25MTECH14003

**Importing required Library**

In [ ]:
import numpy as np
np.set_printoptions(precision=10, suppress=True)

**Load Linear Programming Instance**

In [ ]:
def check_testcase(csv_path: str):
    data = np.genfromtxt(csv_path, delimiter=",", dtype=float)
    n = data.shape[1] - 1
    m = data.shape[0] - 2
    z0 = data[0, :n]
    c = data[1, :n]
    A = data[2:, :n]
    b = data[2:, n]
    return A, b, c, z0, m, n

**Active Constraints Identification**

In [ ]:
def activeness(A, b, z, tol=1e-9):
    return np.where(np.abs(b - A @ z) <= tol)[0]

**Calculate Nullspace**

In [ ]:
def check_nullspace(M, tol=1e-12):
    if M.size == 0:
        return np.eye(M.shape[1])
    U, s, Vt = np.linalg.svd(M, full_matrices=True)
    r = (s > tol).sum()
    return Vt[r:].T

**Main function**

In [ ]:
def find_objective(csv_path: str, verbose=True, tol=1e-9, max_iter=5000):

    A, b, c, z, m, n = check_testcase(csv_path)

    # Feasibility check
    if not np.all(A @ z <= b + tol):
        raise ValueError("Initial point is not feasible (violates A z <= b).")

    # Printing details of given testcase
    if verbose:
        print("Initial Problem Setup\n")
        print(f"Variables (n):   {n}")
        print(f"Constraints (m): {m}")
        print(f"Start Point (z): {z}")
        print(f"Start Value (f): {c @ z:.10f}")
        print(f"Cost Vector (c): {c}")
        print("Constraint Matrix (A):")
        print(A)
        print(f"Constraint RHS (b): {b}\n\n")

    path = [z.copy()]
    vals = [float(c @ z)]

    visited_states = set()

    # Move to first vertex
    for _ in range(5 * (m + n)):
        I = activeness(A, b, z, tol)
        if len(I) >= n:
            if verbose:
                print(f"First vertex we get is: z = {z}, f = {c @ z:.10f}")
            break

        N = check_nullspace(A[I, :], tol) if len(I) > 0 else np.eye(n)
        d = N @ (N.T @ c)
        if np.linalg.norm(d) < tol:
            if N.shape[1] > 0:
                d = N[:, 0]
            else:
                break

        Ad = A @ d
        slack = b - A @ z
        alpha, j_in = np.inf, -1
        for j in range(m):
            if Ad[j] > tol:
                step = slack[j] / Ad[j]
                if step < alpha:
                    alpha = step
                    j_in = j

        # Check for unboundedness
        if not np.isfinite(alpha):
            if verbose:
                print("\nUNBOUNDED: objective → +∞")
            return None

        z = z + alpha * d
        path.append(z.copy())
        vals.append(float(c @ z))

    # Edge walking with cycle detection
    degenerate_count = 0
    last_obj_value = c @ z

    for it in range(max_iter):
        I = activeness(A, b, z, tol)

        state_sig = (tuple(np.round(z, 8)), tuple(sorted(I)))

        # Check for cycle
        if state_sig in visited_states:
            if verbose:
                print(f"\nCYCLE DETECTED at iteration {it+1}")
                print("Algorithm is cycling through same vertices (degenerate problem)")
                print(f"Current vertex: {z}")
                print(f"Optimal value found: {c @ z:.10f}")
            break
        visited_states.add(state_sig)

        if len(I) >= n:
            # Bland's rule: select basis using smallest indices
            I_sel = []
            R = None
            I_sorted = sorted(I)

            for r in I_sorted:
                if R is None:
                    R = A[[r], :]
                    I_sel = [r]
                else:
                    candidate = np.vstack([R, A[r, :]])
                    if np.linalg.matrix_rank(candidate, tol) > np.linalg.matrix_rank(R, tol):
                        R = candidate
                        I_sel.append(r)
                        if len(I_sel) == n:
                            break

            if len(I_sel) < n:
                if verbose:
                    print("\nCould not find n linearly independent constraints. Stopping.")
                break

            A_I = A[I_sel, :]

            try:
                lam = np.linalg.solve(A_I.T, c)
            except np.linalg.LinAlgError:
                if verbose:
                    print("\nSingular matrix encountered. Stopping.")
                break

            # Check optimality
            if np.all(lam >= -tol):
                if verbose:
                    print("\nOptimal vertex reached.")
                break

            # Bland's rule: Choose leaving variable with smallest index
            neg_indices = [i for i, val in enumerate(lam) if val < -tol]
            if not neg_indices:
                if verbose:
                    print("\nOptimal vertex reached.")
                break
            k = neg_indices[0]  # Smallest index

            e = np.zeros(len(I_sel))
            e[k] = 1.0
            d = -np.linalg.solve(A_I, e)
        else:
            # Fewer than n active constraints
            N = check_nullspace(A[I, :], tol)
            if N.size == 0:
                if verbose:
                    print("\nEmpty nullspace. Stopping.")
                break
            d = N @ (N.T @ c)
            if np.linalg.norm(d) < tol:
                if verbose:
                    print("\nNo improving direction found. Optimal.")
                break

        # Find blocking constraint
        Ad = A @ d
        slack = b - A @ z

        blocking_candidates = []
        for j in range(m):
            if Ad[j] > tol:
                step = slack[j] / Ad[j]
                blocking_candidates.append((step, j))

        if not blocking_candidates:
            if verbose:
                print("\nUNBOUNDED: objective → +∞")
            return None

        blocking_candidates.sort(key=lambda x: (x[0], x[1]))
        alpha, j_in = blocking_candidates[0]

        # Check for degeneracy
        if np.abs(alpha) < tol:
            degenerate_count += 1
            if verbose and degenerate_count == 1:
                print(f"\nDEGENERATE pivot detected (α ≈ 0)")

        z = z + alpha * d
        new_obj = c @ z

        if np.abs(new_obj - last_obj_value) < tol:
            degenerate_count += 1
            if degenerate_count > 20:
                if verbose:
                    print(f"\n Too many degenerate pivots ({degenerate_count}). Likely cycling.")
                    print(f"Stopping at vertex: {z}, f = {new_obj:.10f}")
                break
        else:
            degenerate_count = 0
            last_obj_value = new_obj

        path.append(z.copy())
        vals.append(float(new_obj))

        if verbose and j_in >= 0:
            print(f"Entered constraint {j_in}; z={z}, f={new_obj:.10f}")

    if verbose:
        print(f"Final optimal vertex: {z}")
        print(f"Optimal value: {c @ z:.10f}")
        print(f"\nTotal vertices visited: {len(path)}")
        if len(path) <= 20:
            print("\nSequence of vertices and objective values:")
            for i, (zi, fi) in enumerate(zip(path, vals), start=1):
                print(f"{i}. {zi.tolist()}   f = {fi:.10f}")
        else:
            print("\nFirst 10 vertices:")
            for i in range(min(10, len(path))):
                print(f"{i+1}. {path[i].tolist()}   f = {vals[i]:.10f}")
            print(f"\n... ({len(path) - 20} vertices omitted) ...\n")
            print("Last 10 vertices:")
            for i in range(max(10, len(path)-10), len(path)):
                print(f"{i+1}. {path[i].tolist()}   f = {vals[i]:.10f}")
        return None

    return z, path, vals

**Run Solver**

In [ ]:
find_objective("/content/Testcase6.csv", verbose=True)

Initial Problem Setup

Variables (n):   2
Constraints (m): 4
Start Point (z): [0. 1.]
Start Value (f): 7.0000000000
Cost Vector (c): [3. 7.]
Constraint Matrix (A):
[[ 1.   0. ]
 [ 1.  -1. ]
 [ 2.  -1. ]
 [ 0.5 -1. ]]
Constraint RHS (b): [ 5.  1.  4. -1.]


First vertex we get is: z = [3.3333333333 2.6666666667], f = 28.6666666667
Entered constraint 0; z=[5. 6.], f=57.0000000000

UNBOUNDED: objective → +∞
